# MediLacra — MPI Snapshot (from HL7 Parser keys)
**Purpose**: Build a Master Patient Index (MPI) snapshot by linking records using identifiers already extracted by your HL7 Parser (e.g., MRN, account number, visit number, accession/order filler numbers).

**Assumptions**:
- Your HL7 Parser produces a table/dataframe of keys with columns like `key_type`, `key_value`, and an entity column (e.g., `case_id`, `visit_number`, or `message_control_id`) that we can cluster on.
- If your parser wrote files, point `INPUT_PATH` to that artifact. Otherwise, you can set `keys_df` in-memory.

**Outputs**:
- A dataframe `mpi_snapshot` with columns: `entity_id`, `cluster_id`, `is_unlinked`, `run_timestamp`.
- Optionally, it can be saved to Parquet/CSV for downstream tools.


## 0) Environment & Dependencies

In [1]:
# If needed, install dependencies (uncomment if running in a fresh env)
#%pip install networkx pandas pyarrow fastparquet
import pandas as pd
import numpy as np
import networkx as nx
from datetime import datetime
import os, re, hashlib

RUN_TIMESTAMP = pd.Timestamp.utcnow().tz_localize(None)
print("Run timestamp (UTC):", RUN_TIMESTAMP)


Run timestamp (UTC): 2025-11-17 17:12:00.663315


## 1) Configure Inputs

In [2]:
# Choose one column below to represent the **entity** you want clustered.
# For example: 'case_id' (preferred), fallback to 'visit_number' or 'message_control_id'.
ENTITY_COL_CANDIDATES = ["patient_ids_norm","visit_number", "account_number", "message_control_id"]
IDENTIFIER_COLS = ["key_type", "key_value"]  # from your HL7 Parser

# If your HL7 Parser saved a file, set INPUT_PATH accordingly; else leave None and define `keys_df` manually below.
INPUT_PATH = r"C:\Data Generator\output\parsed"    # e.g., '/mnt/data/hl7_parser_keys.parquet' or 'keys.csv'
INPUT_FORMAT = 'csv'  # 'parquet' | 'csv'

# Optional: keep only certain key types (leave empty to use all present)
ALLOWED_KEY_TYPES = [
    # examples (edit to match your parser's key_type values)
    "MRN","VISIT_NUMBER","ACCOUNT_NUMBER","HOSPITAL_ACCOUNT_RECORD",
    "ACCESSION_NUMBER","ORDER_FILLER_NUMBER","PLACER_ORDER_NUMBER"
]
FILTER_KEY_TYPES = False  # set True to enforce ALLOWED_KEY_TYPES


### 1a) Load Keys

In [3]:
# Option A: Load from a saved artifact
keys_df = None
if INPUT_PATH:
    if INPUT_FORMAT == "parquet":
        keys_df = pd.read_parquet(INPUT_PATH)
    elif INPUT_FORMAT == "csv":
        keys_df = pd.read_csv(INPUT_PATH)
    else:
        raise ValueError("Please set INPUT_FORMAT to 'parquet' or 'csv'.")

# Option B: If not loaded from disk, attempt to reconstruct from an existing in-memory table (`hl7_keys`), or create a demo
if keys_df is None:
    try:
        # If your session already has a DataFrame named hl7_keys (from your parser), use it:
        keys_df = hl7_keys.copy()
        print("Loaded keys from in-memory `hl7_keys`.")
    except NameError:
        # --- Demo scaffold (delete/replace) ---
        print("No input found; creating a small demo set. Replace this with your parser output.")
        keys_df = pd.DataFrame({
            "message_control_id": ["A1","A2","B1","B2","C1"],
            "key_type": ["MRN","VISIT_NUMBER","MRN","ACCOUNT_NUMBER","MRN"],
            "key_value": ["0012345","V-777","0012345","ACC-9","9999999"],
            "case_id": [101,102,103,103,999],
            "visit_number": ["V-1","V-2","V-3","V-3","V-9"],
            "account_number": ["ACC-1","ACC-2","ACC-3","ACC-3","ACC-9"],
        })
keys_df.head(10)


PermissionError: [Errno 13] Permission denied: 'C:\\Data Generator\\output\\parsed'

### 1b) Choose `entity_id` & normalize identifiers

In [ ]:
# Pick the first available candidate as the entity column.
entity_col = None
for c in ENTITY_COL_CANDIDATES:
    if c in keys_df.columns:
        entity_col = c
        break
if entity_col is None:
    raise ValueError(f"None of {ENTITY_COL_CANDIDATES} were found in keys_df columns: {list(keys_df.columns)}")

print("Using entity column:", entity_col)

# Normalize id values: strip leading zeros, standardize case, remove trivial values
def norm_id(x):
    if pd.isna(x): 
        return None
    s = str(x).strip()
    s = re.sub(r'^0+', '', s)  # drop leading zeros
    s = s.upper()
    if s in {"", "0", "UNKNOWN", "N/A", "NULL"}:
        return None
    return s

df = keys_df.copy()
df["id_value"] = df["key_value"].map(norm_id)
df["id_type"]  = df["key_type"].str.upper().str.strip()

if FILTER_KEY_TYPES and ALLOWED_KEY_TYPES:
    df = df[df["id_type"].isin([t.upper() for t in ALLOWED_KEY_TYPES])]

df = df[[entity_col, "id_type", "id_value"]].dropna(subset=[entity_col, "id_value"]).drop_duplicates()
df.rename(columns={entity_col: "entity_id"}, inplace=True)

print("Rows after normalization:", len(df))
df.head(10)


## 2) Build the Link Graph & Connected Components

In [ ]:
# Build edges by linking all entity_ids that share the same id_value.
# Strategy: for each id_value, all entities with that value belong to one clique; we add edges between consecutive pairs.

# Group entities per identifier value
groups = df.groupby("id_value")["entity_id"].apply(list)

edges = []
for ent_list in groups:
    if len(ent_list) > 1:
        # connect in a chain to avoid dense cliques (same component result)
        for a, b in zip(ent_list[:-1], ent_list[1:]):
            edges.append((a, b))

print("Edge count:", len(edges))

# Build the graph
G = nx.Graph()
G.add_edges_from(edges)

# Ensure isolated nodes (no edges) are still represented
all_entities = pd.Index(df["entity_id"].unique())
G.add_nodes_from(all_entities)

# Connected components -> cluster_id assignment
components = list(nx.connected_components(G))
print("Component count:", len(components))

records = []
for i, comp in enumerate(components):
    cluster_id = f"MPI_{i:06d}"
    for ent in comp:
        records.append({"entity_id": ent, "cluster_id": cluster_id, "is_unlinked": (len(comp)==1)})
mpi_snapshot = pd.DataFrame.from_records(records).sort_values(["cluster_id", "entity_id"]).reset_index(drop=True)
mpi_snapshot["run_timestamp"] = RUN_TIMESTAMP

mpi_snapshot.head(20)


## 3) (Optional) Enrich with Preferred Person-Level Identifiers

In [ ]:
# If you want to compute person-level attributes (e.g., a canonical MRN per cluster), compute them here.
# Example: choose the smallest MRN string in each cluster (demo logic; adjust per policy).

# Map entity_id -> MRN values present
mrn_df = keys_df.copy()
if "MRN" in df["id_type"].unique():
    mrn_df["MRN_norm"] = mrn_df.loc[mrn_df["key_type"].str.upper()=="MRN", "key_value"].map(lambda x: x)
else:
    mrn_df["MRN_norm"] = None

# Join MRNs for entities we clustered
mrn_map = (df[df["id_type"]=="MRN"][["entity_id","id_value"]]
           .rename(columns={"id_value":"MRN_norm"}))

mpi_enriched = mpi_snapshot.merge(mrn_map, on="entity_id", how="left")

# Pick a representative MRN per cluster (example policy)
rep = (mpi_enriched.dropna(subset=["MRN_norm"])
       .groupby("cluster_id")["MRN_norm"].min()
       .rename("canonical_mrn"))
mpi_enriched = mpi_enriched.merge(rep, on="cluster_id", how="left")

mpi_enriched.head(20)


## 4) Save Artifacts

In [ ]:
OUTPUT_DIR = "/mnt/data/medilacra_mpi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpi_snapshot_path_parquet = os.path.join(OUTPUT_DIR, "mpi_snapshot.parquet")
mpi_snapshot_path_csv     = os.path.join(OUTPUT_DIR, "mpi_snapshot.csv")

mpi_snapshot.to_parquet(mpi_snapshot_path_parquet, index=False)
mpi_snapshot.to_csv(mpi_snapshot_path_csv, index=False)

print("Saved:")
print(" -", mpi_snapshot_path_parquet)
print(" -", mpi_snapshot_path_csv)


## 5) Quick Stats

In [ ]:
total_entities = mpi_snapshot["entity_id"].nunique()
total_clusters = mpi_snapshot["cluster_id"].nunique()
unlinked = mpi_snapshot["is_unlinked"].sum()
linked = total_entities - unlinked
print(f"Entities: {total_entities:,} | Clusters: {total_clusters:,} | Linked: {linked:,} | Unlinked: {unlinked:,}")
